# Capítulo 3 — Decomposição de Séries Temporais
**Livro:** Forecasting: Principles and Practice, the Pythonic Way  
**Pesquisa:** Previsão de Demanda com Séries Hierárquicas | UPE

---

## Seções do capítulo
- 3.1 Transformações e ajustes (calendário, populacional, inflação, Box-Cox)
- 3.2 Componentes da série temporal (aditivo vs multiplicativo)
- 3.3 Médias móveis (base do baseline que seus modelos precisam superar)
- 3.4 Decomposição clássica (e por que não usar)
- 3.5 X-11 e SEATS (referência histórica)
- 3.6 STL — o método que você vai usar na pesquisa

---

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import STL
import warnings
warnings.filterwarnings('ignore')

# instale se necessário:
# !pip install coreforecast utilsforecast --quiet
from utilsforecast.plotting import plot_series

plt.style.use('ggplot')
plt.rcParams.update({'figure.figsize': (10, 4), 'figure.dpi': 100})
print('Ambiente pronto.')

---
## 3.1 Transformações e Ajustes

Simplificar a série antes de decompor melhora a precisão. Quatro tipos de ajuste:

| Tipo | Quando usar | Como fazer |
|------|-------------|------------|
| Calendário | Dados mensais com dias úteis variáveis | Dividir por número de dias úteis |
| Populacional | Dados influenciados por crescimento da população | Dividir pela população |
| Inflação | Séries financeiras de longo prazo | Dividir pelo CPI e multiplicar pelo CPI base |
| Box-Cox | Sazonalidade cresce com o nível da série | boxcox_lambda() + boxcox() |

In [ ]:
# ajuste populacional: PIB per capita da Austrália
global_economy = pd.read_csv('data/global_economy.csv')
australia = (
    global_economy
    .loc[lambda x: x['unique_id'] == 'Australia']
    .assign(gdp_per_capita=lambda x: x['GDP'] / x['Population'])
)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(australia['ds'], australia['GDP'], color='steelblue')
axes[0].set_title('PIB total — Austrália (pode crescer só por população)')
axes[0].set_ylabel('USD')

axes[1].plot(australia['ds'], australia['gdp_per_capita'], color='darkorange')
axes[1].set_title('PIB per capita — Austrália (crescimento real)')
axes[1].set_ylabel('USD por habitante')

plt.tight_layout()
plt.show()

# O PIB per capita revela o crescimento REAL, descontando o efeito da população
# Na sua pesquisa: demanda por loja normalizada por fluxo de clientes = mesmo conceito

In [ ]:
# transformação Box-Cox — estabiliza variância sazonal
try:
    from coreforecast.scalers import boxcox, boxcox_lambda

    aus_production = pd.read_csv('data/aus_production.csv', parse_dates=['ds'])
    aus_gas = (
        aus_production[['ds', 'Gas']]
        .rename(columns={'Gas': 'y'})
        .assign(unique_id='Gas')
        .dropna()
    )

    y = aus_gas['y'].to_numpy()
    optim_lambda = boxcox_lambda(y, method='guerrero', season_length=4)
    print(f'Lambda ótimo: {optim_lambda:.4f}')
    print(f'Lambda próximo de 0: usar log | próximo de 1: sem transformação necessária')

    aus_gas['y_transformed'] = boxcox(y, optim_lambda)

    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    axes[0].plot(aus_gas['ds'], aus_gas['y'], color='steelblue')
    axes[0].set_title('Produção de gás — original (variância sazonal cresce)')

    axes[1].plot(aus_gas['ds'], aus_gas['y_transformed'], color='darkorange')
    axes[1].set_title(f'Produção de gás — Box-Cox (λ={optim_lambda:.2f}) — variância estabilizada')

    plt.tight_layout()
    plt.show()

except ImportError:
    print('coreforecast não instalado. Rode: pip install coreforecast')
    print('Alternativa simples: y_log = np.log(y)')

    aus_production = pd.read_csv('data/aus_production.csv', parse_dates=['ds'])
    aus_gas = aus_production[['ds', 'Gas']].dropna()

    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    axes[0].plot(aus_gas['ds'], aus_gas['Gas'], color='steelblue')
    axes[0].set_title('Produção de gás — original')

    axes[1].plot(aus_gas['ds'], np.log(aus_gas['Gas']), color='darkorange')
    axes[1].set_title('Produção de gás — log (aproximação Box-Cox com λ≈0)')

    plt.tight_layout()
    plt.show()

---
## 3.2 Componentes da Série Temporal

**Decomposição aditiva:** `y(t) = S(t) + T(t) + R(t)`  
Usar quando a amplitude sazonal é constante ao longo do tempo.

**Decomposição multiplicativa:** `y(t) = S(t) × T(t) × R(t)`  
Usar quando os picos sazonais crescem proporcionalmente ao nível da série.

**Como identificar qual usar:** olhe o gráfico temporal. Se os vales e picos ficam maiores junto com a série, é multiplicativo. Se ficam constantes, é aditivo.

In [ ]:
# STL na série de emprego no varejo dos EUA
us_employment = pd.read_csv('data/us_employment.csv', parse_dates=['ds'])
us_retail = us_employment.loc[
    lambda x: (x['unique_id'] == 'Retail Trade') & (x['ds'] >= '1990')
]

# decomposição STL
stl = STL(us_retail['y'], period=12)
res = stl.fit()

# monta DataFrame com os componentes
dcmp = pd.DataFrame({
    'ds':        us_retail['ds'].values,
    'data':      us_retail['y'].values,
    'trend':     res.trend,
    'seasonal':  res.seasonal,
    'remainder': res.resid,
})

# dado sazonalmente ajustado
dcmp['adjusted'] = dcmp['data'] - dcmp['seasonal']

print('Primeiras linhas da decomposição:')
print(dcmp.head().to_string())

In [ ]:
# visualiza os 4 componentes
fig, axes = plt.subplots(4, 1, sharex=True, figsize=(10, 8))

axes[0].plot(dcmp['ds'], dcmp['data'], color='steelblue', linewidth=1)
axes[0].set_title('Emprego no varejo = tendência + sazonal + resíduo', loc='left', fontsize=10)
axes[0].set_ylabel('Empregados')

axes[1].plot(dcmp['ds'], dcmp['trend'], color='darkorange', linewidth=1.5)
axes[1].set_ylabel('Tendência')

axes[2].plot(dcmp['ds'], dcmp['seasonal'], color='green', linewidth=1)
axes[2].set_ylabel('Sazonal')

axes[3].plot(dcmp['ds'], dcmp['remainder'], color='gray', linewidth=0.8)
axes[3].axhline(0, color='black', linestyle='--', linewidth=0.5)
axes[3].set_ylabel('Resíduo')

fig.suptitle('Decomposição STL — Emprego no Varejo EUA', y=1.01)
plt.tight_layout()
plt.show()

# O resíduo deveria ser ruído branco
# Se houver padrão no resíduo, o modelo não capturou tudo

In [ ]:
# dado sazonalmente ajustado vs original
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(dcmp['ds'], dcmp['data'], color='gray', alpha=0.7, label='Original', linewidth=1)
ax.plot(dcmp['ds'], dcmp['adjusted'], color='steelblue', linewidth=1.5,
        label='Sazonalmente ajustado')
ax.set_title('Emprego no Varejo — Original vs Sazonalmente Ajustado')
ax.set_ylabel('Empregados (mil)')
ax.legend()
plt.tight_layout()
plt.show()

# O dado ajustado ainda contém tendência + resíduo
# Por isso as quedas de 2000 e 2008 ainda aparecem — são eventos reais, não sazonais

---
## 3.3 Médias Móveis

Médias móveis estimam a tendência-ciclo suavizando o ruído e a sazonalidade.

**Regras para escolher o tipo:**
- Dados anuais: qualquer MA ímpar (3, 5, 7)
- Dados trimestrais (m=4): usar **2×4-MA**
- Dados mensais (m=12): usar **2×12-MA**
- Dados diários com sazonalidade semanal (m=7): usar **7-MA**

**Por que center=True é obrigatório?** Sem ele, a MA olha só para o passado (MA causal) — isso introduz atraso na estimativa de tendência.

In [ ]:
# média móvel de diferentes ordens
aus_exports = (
    global_economy
    .loc[lambda x: x['unique_id'] == 'Australia']
    .assign(
        MA_3=lambda x: x['Exports'].rolling(3,  center=True).mean(),
        MA_5=lambda x: x['Exports'].rolling(5,  center=True).mean(),
        MA_9=lambda x: x['Exports'].rolling(9,  center=True).mean(),
    )
)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for ax, col, label in zip(axes,
                           ['MA_3', 'MA_5', 'MA_9'],
                           ['3-MA (mais sensível)', '5-MA (equilíbrio)', '9-MA (mais suave)']):
    ax.plot(aus_exports['ds'], aus_exports['Exports'],
            color='gray', alpha=0.5, linewidth=1, label='Exportações')
    ax.plot(aus_exports['ds'], aus_exports[col],
            color='darkorange', linewidth=2, label=label)
    ax.set_title(label)
    ax.set_xlabel('Ano')
    ax.legend(fontsize=8)

fig.suptitle('Efeito da ordem da Média Móvel — Exportações Australianas')
plt.tight_layout()
plt.show()

# Quanto maior m, mais suave — mas mais observações são perdidas nas bordas

In [ ]:
# 2×12-MA para dados mensais — elimina sazonalidade anual
df_retail = us_retail.copy()
df_retail['12-MA']   = df_retail['y'].rolling(window=12, center=True).mean()
df_retail['2x12-MA'] = df_retail['12-MA'].rolling(window=2, center=True).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_retail['ds'], df_retail['y'], color='gray', alpha=0.6, linewidth=1, label='Dados')
ax.plot(df_retail['ds'], df_retail['2x12-MA'], color='darkorange',
        linewidth=2, label='2×12-MA (tendência estimada)')
ax.set_title('2×12-MA — Elimina sazonalidade mensal e revela tendência')
ax.set_ylabel('Empregados (mil)')
ax.legend()
plt.tight_layout()
plt.show()

# A 2x12-MA é quase idêntica ao componente de tendência do STL
# Diferença: STL usa Loess (mais sofisticado) e permite sazonalidade variável

---
## 3.4 Decomposição Clássica vs STL

### Por que NOT usar decomposição clássica

1. Sem estimativa de tendência nos primeiros e últimos k períodos
2. Suaviza demais mudanças bruscas
3. **Sazonalidade constante ao longo do tempo** — problema crítico para dados de demanda que evoluem
4. Não é robusta a outliers

### Use STL

- Aceita qualquer frequência
- Sazonalidade pode variar ao longo do tempo
- Controlável pelo usuário
- Robusta com `robust=True`

In [ ]:
# STL padrão vs STL ajustado — diferença prática
fig, axes = plt.subplots(2, 4, figsize=(14, 7), sharex=True)

# STL padrão
stl_default = STL(us_retail['y'], period=12)
res_default = stl_default.fit()

# STL ajustado (mais flexível, robusto)
stl_robust = STL(us_retail['y'], period=12, seasonal=13, trend=21, robust=True)
res_robust = stl_robust.fit()

ds = us_retail['ds'].values
y  = us_retail['y'].values

labels = ['Dados', 'Tendência', 'Sazonal', 'Resíduo']
colors = ['steelblue', 'darkorange', 'green', 'gray']

for i, (comp_default, comp_robust, label, color) in enumerate(zip(
    [y, res_default.trend, res_default.seasonal, res_default.resid],
    [y, res_robust.trend, res_robust.seasonal, res_robust.resid],
    labels, colors
)):
    axes[0, i].plot(ds, comp_default, color=color, linewidth=1)
    axes[0, i].set_title(label, fontsize=9)
    axes[1, i].plot(ds, comp_robust, color=color, linewidth=1)

axes[0, 0].set_ylabel('STL padrão', fontsize=9)
axes[1, 0].set_ylabel('STL robusto\n(seasonal=13, trend=21)', fontsize=9)

fig.suptitle('STL padrão vs STL ajustado — Emprego no Varejo EUA')
plt.tight_layout()
plt.show()

# A diferença aparece principalmente na tendência:
# STL robusto captura melhor a queda de 2008 (crise financeira)
# STL padrão suaviza demais esse evento

---
## 3.6 STL na sua pesquisa — aplicação prática

Exemplo real com o dataset de turismo australiano (que você conhece desde o cap. 1).
Decompor cada série da hierarquia revela comportamentos diferentes entre níveis.

In [ ]:
# STL aplicado a múltiplas séries da hierarquia de turismo
tourism = pd.read_csv('data/aus_tourism.csv', parse_dates=['ds'])

# extrai componentes hierárquicos
tourism[['Region', 'State', 'Purpose']] = \
    tourism['unique_id'].str.rsplit('-', n=2, expand=True)

# total Australia (nível mais alto da hierarquia)
total_aus = tourism.groupby('ds')['y'].sum().reset_index()

# um estado específico
nsw = tourism[tourism['State'] == 'New South Wales'].groupby('ds')['y'].sum().reset_index()

# decompõe as duas séries
def decompor_stl(serie_y, period=4):
    """Aplica STL e retorna DataFrame com componentes."""
    stl = STL(serie_y, period=period, robust=True)
    res = stl.fit()
    return pd.DataFrame({
        'trend': res.trend,
        'seasonal': res.seasonal,
        'remainder': res.resid
    })

dcmp_aus = decompor_stl(total_aus['y'], period=4)
dcmp_nsw = decompor_stl(nsw['y'], period=4)

# visualiza comparação
fig, axes = plt.subplots(3, 2, figsize=(14, 8), sharex=True)

componentes = ['trend', 'seasonal', 'remainder']
nomes = ['Tendência', 'Sazonalidade', 'Resíduo']
cores = ['darkorange', 'green', 'gray']

for i, (comp, nome, cor) in enumerate(zip(componentes, nomes, cores)):
    axes[i, 0].plot(total_aus['ds'], dcmp_aus[comp], color=cor, linewidth=1.2)
    axes[i, 0].set_ylabel(nome, fontsize=9)
    axes[i, 1].plot(nsw['ds'], dcmp_nsw[comp], color=cor, linewidth=1.2)

axes[0, 0].set_title('Total Austrália (nível 1)', fontsize=10)
axes[0, 1].set_title('New South Wales (nível 2)', fontsize=10)

fig.suptitle('STL Hierárquico — Componentes por Nível', y=1.01)
plt.tight_layout()
plt.show()

# O padrão sazonal do estado deve ser coerente com o nacional
# Diferenças no resíduo revelam comportamentos específicos de cada nível

In [ ]:
# força da sazonalidade e da tendência — métricas do cap. 4
# antecipando: útil para entender qual série precisa de modelo mais sofisticado

def forca_sazonalidade(decomposicao):
    """Calcula força da sazonalidade. Valor entre 0 e 1."""
    var_resid = decomposicao['remainder'].var()
    var_saz_resid = (decomposicao['seasonal'] + decomposicao['remainder']).var()
    return max(0, 1 - var_resid / var_saz_resid)

def forca_tendencia(decomposicao):
    """Calcula força da tendência. Valor entre 0 e 1."""
    var_resid = decomposicao['remainder'].var()
    var_tend_resid = (decomposicao['trend'] + decomposicao['remainder']).var()
    return max(0, 1 - var_resid / var_tend_resid)

print('Total Austrália:')
print(f'  Força da sazonalidade: {forca_sazonalidade(dcmp_aus):.3f}')
print(f'  Força da tendência:    {forca_tendencia(dcmp_aus):.3f}')
print()
print('New South Wales:')
print(f'  Força da sazonalidade: {forca_sazonalidade(dcmp_nsw):.3f}')
print(f'  Força da tendência:    {forca_tendencia(dcmp_nsw):.3f}')
print()
print('Referência: > 0.6 = forte | < 0.4 = fraco')
print('Cap. 4 vai formalizar essas métricas como features de séries temporais')

---
## Consolidação — O que você precisa saber do cap. 3

| Conceito | Aplicação na pesquisa |
|----------|------------------------|
| Box-Cox / log | Pré-processar séries de demanda com sazonalidade multiplicativa |
| Decomposição aditiva | Quando amplitude sazonal é constante (demanda estável) |
| Decomposição multiplicativa | Quando amplitude sazonal cresce (demanda em expansão) |
| STL period=12 | Dados mensais com sazonalidade anual |
| STL period=4 | Dados trimestrais com sazonalidade anual |
| robust=True | Quando dados têm outliers (paradas de produção, disrupções) |
| Dado ajustado | Referência para análise de tendência — remove efeito sazonal |
| Resíduo ≈ ruído branco | Critério de qualidade do modelo — se não for, o modelo falhou |

---

## Próximo: Capítulo 4 — Features de Séries Temporais

Métricas resumidas de uma série (força da sazonalidade, força da tendência, entropia, ACF1).  
Usadas para clustering de séries e seleção automática de modelos — diretamente aplicável ao seu dataset hierárquico.

---
*Notebook de estudo — Previsão de Demanda com Séries Hierárquicas | UPE*